# Libs Imports

In [1]:
import requests
import pandas as pd
from fredapi import Fred
import yfinance as yf
import seaborn as sns
import matplotlib.pyplot as plt
import time
from json import JSONDecodeError
from requests.exceptions import RequestException

# Base Indicadores Brasil

- SELIC
- IPCA
- PIB
- Balança Comercial
    - Importações
    - Exportações

In [2]:
BASES_INDICES = {
    "Selic": 432,
    "IPCA": 433,
    "PIB": 4380,
    "Exportacoes": 22708,
    "Importacoes": 22709,
}

data_inicial = "01/05/2016"
data_final = "01/05/2026"

base_consolidada_BR = pd.DataFrame(
    index=pd.date_range(pd.to_datetime(data_inicial, dayfirst=True), pd.to_datetime(data_final, dayfirst=True), freq="D"),
    columns=["valor_inicial"]
)

dict_of_dataframes = {}


def fetch_json_with_retry(url, max_retries=3, delay=2):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except (RequestException, JSONDecodeError) as exc:
            last_error = exc
            if attempt == max_retries:
                raise RuntimeError(f"Falha ao obter dados após {max_retries} tentativas: {exc}") from exc
            time.sleep(delay)
            print(f"Tentativa {attempt} falhou: {exc}. Repetindo em {delay} segundos...")


for indice, codigo in BASES_INDICES.items():
    print(f"Processando: {indice} - {codigo}...")
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"
    dados = fetch_json_with_retry(url)
    df = pd.DataFrame(dados)
    df['data'] = pd.to_datetime(df['data'], dayfirst=True)
    df.set_index('data', inplace=True)
    df[indice] = pd.to_numeric(df['valor'])
    df.drop(columns=['valor'], inplace=True)

    dict_of_dataframes[indice] = df
    base_consolidada_BR = base_consolidada_BR.merge(df, how='left', left_index=True, right_index=True)

base_consolidada_BR.drop(columns=['valor_inicial'], inplace=True)

Processando: Selic - 432...
Tentativa 1 falhou: HTTPSConnectionPool(host='api.bcb.gov.br', port=443): Read timed out. (read timeout=30). Repetindo em 2 segundos...
Tentativa 2 falhou: HTTPSConnectionPool(host='api.bcb.gov.br', port=443): Read timed out. (read timeout=30). Repetindo em 2 segundos...
Processando: IPCA - 433...
Processando: PIB - 4380...
Processando: Exportacoes - 22708...
Processando: Importacoes - 22709...


# Base EUA e Mercado

In [3]:
base_consolidada = base_consolidada_BR.copy()

## FRED API

In [4]:
FRED_KEY = '91956fd36b989b465c65ddbce8d24cc6'

In [5]:
fred = Fred(api_key=FRED_KEY)

fed = fred.get_series('FEDFUNDS')

fed = pd.DataFrame(fed, columns=['FEDFUNDS'])

In [6]:
base_consolidada = base_consolidada.merge(fed, how='left', left_index=True, right_index=True)

## Yahoo finance API

In [7]:
BASES_INDICES_GLOBAIS = {
    "IBOV": "^BVSP",
    "SP500": "^GSPC",
    "USD/BRL": "USDBRL=X",
    "Petroleo": "CL=F"
}
start_date = "2016-05-01"
end_date = "2026-05-01"


for indice, ticker in BASES_INDICES_GLOBAIS.items():
    data = yf.download(
        ticker,
        start=start_date,
        end=end_date
    );
    data = data[['Close']].copy()
    data.columns = [indice]
    data.index = pd.to_datetime(data.index)
    dict_of_dataframes[indice] = data
    
    base_consolidada = base_consolidada.merge(data , how='left', left_index=True, right_index=True)

C:\Users\gnlin\AppData\Local\Temp\ipykernel_30416\2481746265.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed
C:\Users\gnlin\AppData\Local\Temp\ipykernel_30416\2481746265.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed
C:\Users\gnlin\AppData\Local\Temp\ipykernel_30416\2481746265.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed
C:\Users\gnlin\AppData\Local\Temp\ipykernel_30416\2481746265.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed


In [8]:
base_consolidada.dropna(axis=0)

,Selic,IPCA,PIB,Exportacoes,Importacoes,FEDFUNDS,IBOV,SP500,USD/BRL,Petroleo
2016-06-01,14.25,0.35,521850.9,16622.9,12991.3,0.38,49013.0,2099.330078,3.5832,49.009998
2016-07-01,14.25,0.52,521952.1,16223.0,11977.9,0.39,52233.0,2102.949951,3.2059,48.990002
2016-08-01,14.25,0.44,530134.3,16925.4,13094.5,0.40,56756.0,2170.840088,3.2462,40.060001
2016-09-01,14.25,0.08,525084.1,15744.2,12221.5,0.40,58236.0,2170.860107,3.2260,43.160000
2016-11-01,14.00,0.18,545558.3,16156.5,11708.1,0.41,63326.0,2111.719971,3.1982,46.669998
...,...,...,...,...,...,...,...,...,...,...
2025-07-01,15.00,0.26,1093459.5,32387.4,25996.9,4.33,139549.0,6198.009766,5.4302,65.449997
2025-08-01,15.00,-0.11,1067005.9,29688.2,24376.7,4.33,132437.0,6238.009766,5.5988,67.330002
2025-10-01,15.00,0.09,1107657.3,31644.4,25754.2,4.09,145517.0,6711.200195,5.3195,61.779999
2025-12-01,15.00,0.33,1084625.6,30856.1,22350.7,3.72,158611.0,6812.629883,5.3354,59.320000


In [9]:
base_consolidada.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3653 entries, 2016-05-01 to 2026-05-01
Freq: D
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Selic        3653 non-null   float64
 1   IPCA         121 non-null    float64
 2   PIB          121 non-null    float64
 3   Exportacoes  121 non-null    float64
 4   Importacoes  121 non-null    float64
 5   FEDFUNDS     121 non-null    float64
 6   IBOV         2484 non-null   float64
 7   SP500        2514 non-null   float64
 8   USD/BRL      2601 non-null   float64
 9   Petroleo     2514 non-null   float64
dtypes: float64(10)
memory usage: 313.9 KB


In [10]:
base_consolidada.describe()

,Selic,IPCA,PIB,Exportacoes,Importacoes,FEDFUNDS,IBOV,SP500,USD/BRL,Petroleo
count,3653.000000,121.000000,1.210000e+02,121.000000,121.000000,121.000000,2484.000000,2514.000000,2601.000000,2514.000000
mean,9.579113,0.413884,7.680279e+05,23389.935537,18933.676033,2.310826,105452.006441,3929.310245,4.639672,65.026114
std,4.210629,0.381275,1.951507e+05,5679.528306,4654.801581,1.896563,28241.674775,1361.121562,0.903338,17.213237
min,2.000000,-0.680000,5.083944e+05,13663.100000,11129.700000,0.050000,48472.000000,2000.540039,3.056100,-37.630001
25%,6.500000,0.210000,5.904864e+05,18403.500000,14491.500000,0.400000,84813.250000,2767.380066,3.775100,52.750000
50%,10.500000,0.360000,7.340240e+05,22545.500000,19666.500000,1.910000,107743.500000,3841.704956,5.007600,64.420002
75%,13.750000,0.590000,9.429100e+05,28946.500000,22927.500000,4.330000,122982.000000,4668.895142,5.367000,75.297503
max,15.000000,1.620000,1.176973e+06,34457.100000,28182.300000,5.330000,198657.000000,7209.009766,6.300000,123.699997


In [ ]:
new_order = list(base_consolidada.columns)
new_order.remove('USD/BRL')
new_order.append('USD/BRL')

base_consolidada = base_consolidada[new_order]

In [ ]:
base_consolidada.to_csv(r'..\dados\bruto\base_consolidada.csv') 